# Week 3 — Regularization

FIN 43800 · Machine Learning in Finance

Andreas Neuhierl

# Where We Are

## The Course So Far, and Next

- **Week 1:** estimation is optimization; in sample is not out of
  sample; more dimensions spread the data thinner.
- **Week 2:** forecasting the equity premium. Almost nothing beats the
  historical mean out of sample, and more predictors made forecasts
  worse.
- **This week:** using many predictors without fitting the noise.
  - **Select** a few, briefly.
  - **Shrink** them all: ridge and the lasso.
  - **Tune:** choose how hard to shrink without peeking.
- **Next:** curves and interactions, trees (Meetings 7 and 8).

## Upload, Then Run Setup

Starting a new Colab session? Colab deletes uploaded files when a
session ends. Download `industries_monthly.csv`, `gw_monthly.csv` and
`gw_annual.csv` from Brightspace, drag them into the Files pane (folder
icon on the left), then run the **Setup** cell below before any other
cell.

## Setup: What It Builds

- **`industries`**: monthly excess returns of 30 industry portfolios
  (food, beer, …, finance, other), July 1926 to December 2025.
- **`market`**: the stock market’s monthly excess return.
- **`monthly`** and **`annual`**: the Goyal–Welch files, as in week 2.
- Source of the first two: Kenneth French’s Data Library.

<!-- -->

- **Your Turn** code is text in this notebook: click **+ Code**, paste
  it, fill the blank, then Shift + Enter.

Run this first, every session.

In [ ]:
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Fetch the one helper file we need from the course repo.
    import subprocess
    subprocess.run(
        ["wget", "-q", "-O", "mlfin_data.py",
         "https://raw.githubusercontent.com/aneuhierl/ml-in-finance-fall2026/main/src/mlfin_data.py"],
        check=True,
    )
else:
    # Find src/mlfin_data.py by walking UP from this notebook's folder.
    here = Path.cwd().resolve()
    src_folders = [folder / "src" for folder in [here, *here.parents]]
    matches = [folder for folder in src_folders if (folder / "mlfin_data.py").is_file()]
    sys.path.insert(0, str(matches[0]) if matches else "../src")
from mlfin_data import resolve_data_path

import time                        # to time a regression
import numpy as np                 # arrays of numbers
import pandas as pd                # tables with columns
import matplotlib.pyplot as plt    # charts
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Kenneth French's 30 industry portfolios (value-weighted) and the market, per month.
# Data: Copyright Eugene F. Fama and Kenneth R. French (French's Data Library).
# Returns are SIMPLE, in decimals per month (0.01 = 1%). rf: the T-bill return of the
# same month. mkt_rf: the market's EXCESS return, already net of rf.
french = pd.read_csv(resolve_data_path("industries_monthly.csv"))
french.index = pd.to_datetime(french["yyyymm"].astype(str), format="%Y%m")
market = french["mkt_rf"]
industry_names = list(french.columns[3:])                 # "Food", "Beer", ..., "Other"
industries = french[industry_names].sub(french["rf"], axis=0)   # EXCESS returns

# The Goyal-Welch files of week 2. equity = stocks minus T-bills, simple, in decimals.
monthly = pd.read_csv(resolve_data_path("gw_monthly.csv"))
monthly.index = pd.to_datetime(monthly["yyyymm"].astype(str), format="%Y%m")
annual = pd.read_csv(resolve_data_path("gw_annual.csv")).set_index("yyyy")
plt.rcParams["font.size"] = 18     # big text for the projector
print(industries.shape, monthly.shape, annual.shape)

## Week 2’s Last Picture

- Pure noise raises the in-sample $R^2$ and lowers the out-of-sample
  $R^2$. Week 3 starts here.

# More Predictors Than Observations

## Finance Has Many Predictors

| Predictors | How many, $p$ |
|------------------------------------|------------------------------------|
| Classic equity-premium predictors (week 2) | 12 |
| Last month’s returns of 30 industries | 30 |
| US macroeconomic series (FRED-MD) | about 130 |
| Published firm characteristics: Cochrane’s “zoo” | hundreds |
| Random combinations of 15 predictors (Kelly, Malamud & Zhou 2024) | 12,000 |

- Against these, the observations $T$ are few: 98 years of annual
  premiums, or 120 months in a 10-year window.
- The books write $n$ for the observations. We write $T$, because later
  $N$ counts assets.

## Week 2’s 1/T Rule, Grown Up

In [ ]:
# A world where every predictor is useless: returns are pure noise, and so are the
# p predictors. Fit OLS on T = 60 months, then score it on 2,000 new months.
generator = np.random.default_rng(3)
T = 60
p_values = [0, 5, 10, 20, 30, 40, 45, 50]
in_sample_r2 = []
out_of_sample_r2 = []
for p in p_values:
    squared_errors_model, squared_errors_mean, r2_values = [], [], []
    for draw in range(300):
        X_fit = generator.normal(size=(T, p))
        y_fit = generator.normal(size=T)
        X_new = generator.normal(size=(2000, p))
        y_new = generator.normal(size=2000)
        if p == 0:                            # no predictors: OLS is the mean
            fitted, forecasts = np.full(T, y_fit.mean()), np.full(2000, y_fit.mean())
        else:
            model = LinearRegression().fit(X_fit, y_fit)
            fitted, forecasts = model.predict(X_fit), model.predict(X_new)
        r2_values.append(1 - np.sum((y_fit - fitted) ** 2) / np.sum((y_fit - y_fit.mean()) ** 2))
        squared_errors_model.append(np.mean((y_new - forecasts) ** 2))
        squared_errors_mean.append(np.mean((y_new - y_fit.mean()) ** 2))
    in_sample_r2.append(100 * np.mean(r2_values))
    out_of_sample_r2.append(100 * (1 - np.mean(squared_errors_model) / np.mean(squared_errors_mean)))

plt.figure(figsize=(10, 2.7))
plt.plot(p_values, in_sample_r2, color="tab:blue", marker="o", linewidth=3)
plt.plot(p_values, out_of_sample_r2, color="tab:orange", marker="o", linewidth=3)
plt.axhline(0, color="black", linewidth=0.8)
plt.text(51, in_sample_r2[-1], "in sample", color="tab:blue", va="center")
plt.text(22, -40, "out of sample:\nfalls off the chart", color="tab:orange", va="center")
plt.ylim(-105, 105)
plt.xlim(-1, 66)
plt.xlabel("number of useless predictors p, with T = 60")
plt.ylabel("R² (%)")
plt.show()

- Error grows by about $p/T$, faster as $p$ nears $T$: in sample the fit
  looks better and better, out of sample it collapses.
- OLS can be “(much) worse than simply forecasting Y by its
  unconditional mean” (Stock & Watson 2006). Read: ISLP 6.4.

## When p ≥ T: Many Perfect Fits

- Two equations, three unknowns:

$$
\begin{aligned}
a + b + c &= 1 \\
a - b &= 0
\end{aligned}
\qquad\Longrightarrow\qquad
b = a,\quad c = 1 - 2a \quad \text{for any } a
$$

- Infinitely many answers. Track the market with the 30 industries: 31
  unknowns (30 weights and an intercept), only 24 months to fit,
  2022–2023.

In [ ]:
# Fit: the market's monthly excess return on the 30 industries' excess returns,
# 24 months (2022-2023). Test: the next 12 months (2024).
fit_months = industries.loc["2022-01":"2023-12"]
test_months = industries.loc["2024-01":"2024-12"]
X_fit = np.column_stack([np.ones(len(fit_months)), fit_months])     # 24 rows, 31 columns
y_fit = market.loc["2022-01":"2023-12"].to_numpy()
X_test = np.column_stack([np.ones(len(test_months)), test_months])
y_test = market.loc["2024-01":"2024-12"].to_numpy()

# 31 unknowns, 24 equations: choose the last 7 weights freely, and the other 24 are then
# pinned down by the 24 months. Different free choices give different exact fits.
for name, free_weight in [("A", 0.0), ("B", 0.2), ("C", 0.5)]:
    weights = np.full(31, free_weight)
    weights[:24] = np.linalg.solve(X_fit[:, :24], y_fit - X_fit[:, 24:] @ weights[24:])
    miss_fit = np.max(np.abs(y_fit - X_fit @ weights))
    miss_test = np.sqrt(np.mean((y_test - X_test @ weights) ** 2))
    print(f"fit {name}: misses 2022-23 by {100 * miss_fit:.1f}%,"
          f" 2024 by {100 * miss_test:.1f}% a month")

## Check: 31 Unknowns, 24 Months

- You regress the market on 30 industries plus an intercept, using 24
  months.
- **Fingers up:** what does OLS do?
  1.  Cannot fit the data at all.
  2.  Finds one best fit with an $R^2$ below 100%.
  3.  Fits all 24 months exactly, in infinitely many ways.
  4.  Returns the historical mean.

# A Biased Estimator Can Predict Better Out of Sample

## What a Miss Costs: The Loss Function

- A forecast $\widehat y$ misses the outcome $y$ by $y - \widehat y$. A
  **loss function** says what a miss costs; fitting minimizes the
  average loss.
- **Squared error** $(y - \widehat y)^2$ is week 1’s SSE and this
  course’s loss almost everywhere. The convenient one: calculus works,
  and it splits into bias and variance.
- Others exist: absolute error $|y - \widehat y|$; the money a forecast
  makes (the portfolio weeks).
- The simplest forecast is one number $c$. Its average squared error
  splits in two:

$$
\frac{1}{T}\sum_{t=1}^{T}(y_t - c)^2
= \frac{1}{T}\sum_{t=1}^{T}(y_t - \bar y)^2 + (\bar y - c)^2
$$

- Only the second part depends on $c$; it is zero at $c = \bar y$. Under
  squared error the **mean** is the best constant forecast: week 2’s
  historical mean. Read: ISLP 2.2.1.

## Shrinking Industry Averages

In [ ]:
def window_averages(start_year):
    """Each industry's average monthly excess return over the 5 years from start_year."""
    return industries.loc[f"{start_year}-01":f"{start_year + 4}-12"].mean()


first_starts = list(range(1970, 2016, 5))            # 1970-74 -> 1975-79, ..., 2015-19 -> 2020-24
ratio_halfway, ratio_grand = [], []
for start in first_starts:
    before, after = window_averages(start), window_averages(start + 5)
    halfway = 0.5 * before + 0.5 * before.mean()      # halfway to the grand mean
    raw = np.mean((after - before) ** 2)              # each industry's own average
    ratio_halfway.append(100 * np.mean((after - halfway) ** 2) / raw)
    ratio_grand.append(100 * np.mean((after - before.mean()) ** 2) / raw)

before, after = window_averages(2005), window_averages(2010)
top_industry = before.idxmax()                        # the best industry of 2005-09
top_rank_after = int(after.rank(ascending=False)[top_industry])   # its rank in 2010-14
low, high = 100 * min(before.min(), after.min()), 100 * max(before.max(), after.max())
figure, axes = plt.subplots(1, 2, figsize=(11, 2.1), gridspec_kw={"width_ratios": [1, 1.4]})
axes[0].scatter(100 * before, 100 * after, color="tab:blue", s=40)
axes[0].plot([low, high], [low, high], color="black", linestyle="--", linewidth=2)
axes[0].annotate(top_industry, (100 * before[top_industry], 100 * after[top_industry]),
                 xytext=(-10, 8), textcoords="offset points", ha="right", va="bottom", fontsize=16)
axes[0].set_xlabel("average, 2005-09 (%)")
axes[0].set_ylabel("average, 2010-14 (%)")
labels = [f"{start % 100:02d}" for start in first_starts]
axes[1].bar(np.arange(10) - 0.2, ratio_halfway, width=0.4, color="tab:orange", label="halfway")
axes[1].bar(np.arange(10) + 0.2, ratio_grand, width=0.4, color="gray", label="all the way")
axes[1].axhline(100, color="black", linewidth=1.5)
axes[1].set_xticks(np.arange(10), labels)
axes[1].set_ylim(0, 135)
axes[1].set_xlabel("first window: 1970, 1975, ..., 2015")
axes[1].set_ylabel("% of own error")
axes[1].legend(fontsize=15, loc="upper center", ncol=2, frameon=False)
plt.subplots_adjust(wspace=0.35)
plt.show()

- Left: the best industry of 2005–09 then fell far down the ranking:
  partly luck. The past 5 years say little about the next 5.
- Right: pulled halfway to the grand mean (the average of all 30), most
  window pairs since 1970 improve.

## Bias and Variance

- **Variance:** how much the fitted line moves from one sample to the
  next.
- **Bias:** the gap between the average fitted line and the truth.
- OLS: no bias, the most variance. Ridge: a small bias, much less
  variance. A huge λ: no variance, all bias.
- **Mean squared error = bias² + variance**, plus noise no model
  removes. A little bias pays if it buys much less variance.

## The Decomposition, Line by Line

- $y = f(x) + \varepsilon$, noise variance $\sigma^2$. $\widehat{f}$ was
  fitted on **other** data, so a new $y$ is independent of it. Use
  $E[Z^2] = \operatorname{Var}(Z) + (EZ)^2$, the loss-function slide’s
  split in expectation, twice.

$$
\begin{aligned}
E\big[(y - \widehat{f})^2\big]
&= E[y^2] + E[\widehat{f}^2] - 2\,E[y]\,E[\widehat{f}] \\
&= \sigma^2 + f^2 + \operatorname{Var}(\widehat{f}) + E[\widehat{f}]^2 - 2f\,E[\widehat{f}] \\
&= \sigma^2 + \operatorname{Var}(\widehat{f}) + \big(f - E[\widehat{f}]\big)^2
\end{aligned}
$$

- The three terms: **noise + variance + bias²**.
- On the **training** data the first line fails: $y$ helped choose
  $\widehat{f}$, so in-sample fit flatters. Read: ISLP 2.2.2.

## A Little Bias, Much Less Variance

In [ ]:
# A world we control: T = 60 observations, p = 30 predictors, and true slopes we know.
# Everything has mean zero, so there is no intercept. Noise variance: 1.
generator = np.random.default_rng(7)
T, p = 60, 30
penalties = np.logspace(-1, 4, 26)                   # lambda from 0.1 to 10,000
x_new = generator.normal(size=(200, p))              # where the forecasts are judged
figure, axes = plt.subplots(1, 2, figsize=(11, 2.1))
best_penalty, ols_error, gain = {}, {}, {}
for axis, (label, signal_r2) in zip(axes, [("strong signal: R² = 50%", 0.50),
                                           ("weak signal: R² = 5%", 0.05)]):
    true_slopes = generator.normal(size=p)
    true_slopes *= np.sqrt(signal_r2 / (1 - signal_r2) / np.sum(true_slopes ** 2))
    truth = x_new @ true_slopes
    forecasts = np.zeros((200, len(penalties), 200))  # simulation x penalty x point
    ols_forecasts = np.zeros((200, 200))              # simulation x point
    for simulation in range(200):
        X = generator.normal(size=(T, p))
        y = X @ true_slopes + generator.normal(size=T)
        ols_forecasts[simulation] = LinearRegression(fit_intercept=False).fit(X, y).predict(x_new)
        for j, penalty in enumerate(penalties):
            ridge = Ridge(alpha=penalty, fit_intercept=False).fit(X, y)   # penalty = lambda
            forecasts[simulation, j] = ridge.predict(x_new)
    bias_squared = ((forecasts.mean(axis=0) - truth) ** 2).mean(axis=1)
    variance = forecasts.var(axis=0).mean(axis=1)
    total = bias_squared + variance
    best_penalty[label] = penalties[np.argmin(total)]
    ols_error[label] = (((ols_forecasts.mean(axis=0) - truth) ** 2).mean()
                        + ols_forecasts.var(axis=0).mean())
    gain[label] = 100 * (1 - total.min() / ols_error[label])
    axis.axhline(ols_error[label], color="gray", linestyle="--", linewidth=2, label="OLS: λ = 0")
    axis.plot(penalties, bias_squared, color="tab:blue", linewidth=3, label="bias²")
    axis.plot(penalties, variance, color="tab:orange", linewidth=3, label="variance")
    axis.plot(penalties, total, color="black", linewidth=3, label="sum")
    axis.plot(best_penalty[label], total.min(), "o", color="black", markersize=10)
    axis.set_xscale("log")
    axis.set_xticks([1, 100, 10000], ["1", "100", "10,000"])
    axis.set_title(label, fontsize=18)
    axis.set_xlabel("ridge penalty λ (log scale)")
axes[0].set_ylabel("error beyond the noise")
axes[1].legend(fontsize=16, loc="center right")
plt.show()

- Dashed: OLS, no bias, all variance. The best λ (dot) cuts its error,
  by far more with the weak signal.
- **The weaker the signal, the harder to shrink.** (ISLP Fig. 6.5)

## How Hard to Shrink

In [ ]:
def window_volatilities(start_year):
    """Each industry's volatility (the standard deviation of its monthly excess return)
    over the 5 years from start_year."""
    return industries.loc[f"{start_year}-01":f"{start_year + 4}-12"].std()


# Weight on the grand mean: 0 = keep each industry's own estimate, 1 = replace every
# estimate by the average of all 30. Pooled over the 10 window pairs.
weights = np.linspace(0, 1, 21)
error_averages, error_volatilities = [], []
for weight in weights:
    total_averages, total_volatilities = 0.0, 0.0
    for start in first_starts:
        before, after = window_averages(start), window_averages(start + 5)
        guess = (1 - weight) * before + weight * before.mean()
        total_averages += np.mean((after - guess) ** 2)
        before_vol, after_vol = window_volatilities(start), window_volatilities(start + 5)
        guess_vol = (1 - weight) * before_vol + weight * before_vol.mean()
        total_volatilities += np.mean((after_vol - guess_vol) ** 2)
    error_averages.append(total_averages)
    error_volatilities.append(total_volatilities)
error_averages = 100 * np.array(error_averages) / error_averages[0]
error_volatilities = 100 * np.array(error_volatilities) / error_volatilities[0]
best_weight_averages = weights[np.argmin(error_averages)]
best_weight_volatilities = weights[np.argmin(error_volatilities)]

plt.figure(figsize=(10, 2.0))
plt.plot(weights, error_averages, color="tab:blue", linewidth=3)
plt.plot(weights, error_volatilities, color="tab:orange", linewidth=3)
plt.plot(best_weight_volatilities, error_volatilities.min(), "o", color="tab:orange",
         markersize=10)
plt.axhline(100, color="black", linewidth=0.8)
plt.text(1.02, error_averages[-1], "averages", color="tab:blue", va="center")
plt.text(1.02, error_volatilities[-1], "volatilities", color="tab:orange", va="center")
plt.xlim(0, 1.28)
plt.xlabel("weight on the grand mean: 0 = no shrinkage, 1 = all the way")
plt.ylabel("squared error, % of none")
plt.show()

- **Averages:** mostly noise: shrink all the way.
- **Volatilities:** persistent: shrink a little (the dot).
- How hard depends on the signal. Meeting 6: no hindsight.

## Check: Bias or Variance?

- Two estimators of next year’s beta:

|     | bias | variance |
|-----|------|----------|
| A   | 0    | 0.06     |
| B   | 0.1  | 0.04     |

- **Fingers up:** which has the lower mean squared error?
  1.  A, because it is unbiased
  2.  A: 0.06 against 0.14
  3.  B
  4.  They tie

## Not Sure? Simulate It

You can always settle a question like these yourself: build a small
world where you know the truth, run it a few hundred times, and look.
Each cell below is one such world. Run it, then change a number and run
it again.

### Pure noise, 40 predictors, 60 months

Returns are pure noise: no predictor has any link to them. OLS is fitted
on 60 months and scored on 2,000 new months. What do the two $R^2$ look
like?

In [ ]:
sim_generator = np.random.default_rng(101)
months, predictors = 60, 40
sim_r2_in, sim_r2_out = [], []
for world in range(200):
    X = sim_generator.standard_normal((months, predictors))   # predictors: noise
    y = sim_generator.standard_normal(months)                 # outcome: noise
    X_new = sim_generator.standard_normal((2000, predictors))
    y_new = sim_generator.standard_normal(2000)
    fit = LinearRegression().fit(X, y)
    sim_r2_in.append(fit.score(X, y))
    # Out of sample R-squared, as in week 2: 1 - (model's errors) / (historical mean's errors).
    sse_model = np.sum((y_new - fit.predict(X_new)) ** 2)
    sse_mean = np.sum((y_new - y.mean()) ** 2)
    sim_r2_out.append(1 - sse_model / sse_mean)
print(f"In sample R-squared:     {100 * np.mean(sim_r2_in):6.0f}%  (average of 200 worlds)")
print(f"Out of sample R-squared: {100 * np.mean(sim_r2_out):6.0f}%")

### Which exact fit?

30 industries and an intercept make 31 unknowns, but there are only 24
months. The cell builds two different sets of weights and checks both
against the 24 months and against one new month.

In [ ]:
sim_generator = np.random.default_rng(102)
X = sim_generator.standard_normal((24, 31))     # 24 months, 31 unknowns
y = sim_generator.standard_normal(24)           # the market's return in each month
weights_a = np.linalg.lstsq(X, y, rcond=None)[0]   # one exact fit
_, _, directions = np.linalg.svd(X)
weights_b = weights_a + 3 * directions[-1]         # move along a direction the 24 months cannot see
new_month = sim_generator.standard_normal(31)
print("Largest miss on the 24 months, fit A:", f"{np.abs(X @ weights_a - y).max():.1e}")
print("Largest miss on the 24 months, fit B:", f"{np.abs(X @ weights_b - y).max():.1e}")
print("Forecast of the new month, fit A:", f"{new_month @ weights_a:6.2f}")
print("Forecast of the new month, fit B:", f"{new_month @ weights_b:6.2f}")

### The best single number

Skewed returns make the mean and the median differ. For each candidate
number $c$, the cell computes the average squared miss and the average
absolute miss.

In [ ]:
sim_generator = np.random.default_rng(103)
returns = sim_generator.exponential(1.0, 200) - 0.5     # a skewed made-up return series
candidates = np.linspace(-1, 3, 401)
squared_miss = [np.mean((returns - c) ** 2) for c in candidates]
absolute_miss = [np.mean(np.abs(returns - c)) for c in candidates]
print("Best c under squared error: ", f"{candidates[np.argmin(squared_miss)]:.2f}")
print("Best c under absolute error:", f"{candidates[np.argmin(absolute_miss)]:.2f}")
print("The mean of the series:     ", f"{returns.mean():.2f}")
print("The median of the series:   ", f"{np.median(returns):.2f}")

### The industry that fell back

30 industries have slightly different true monthly means. In each world
we pick the best industry over the first five years and look up its rank
over the next five.

In [ ]:
sim_generator = np.random.default_rng(104)
true_mean = sim_generator.normal(0.008, 0.002, 30)     # monthly, in decimals (0.008 = 0.8%)
noise_sd = 0.05 / np.sqrt(60)                          # sd of a 5-year average: 5% a month, 60 months
ranks_next, luck = [], []
for world in range(2000):
    first = true_mean + sim_generator.normal(0, noise_sd, 30)    # first 5 years' averages
    second = true_mean + sim_generator.normal(0, noise_sd, 30)   # next 5 years' averages
    best = np.argmax(first)
    ranks_next.append(np.sum(second > second[best]) + 1)         # 1 = best of the 30
    luck.append(first[best] - true_mean[best])
print(f"Average rank of the first period's best industry in the next: {np.mean(ranks_next):.1f} of 30")
print(f"Average luck in its first-period average: {100 * np.mean(luck):.2f}% a month")
print(f"(For reference, a typical 5-year average is off by {100 * noise_sd:.2f}% a month.)")

### Bias and variance, by counting

Two estimators of a quantity whose true value is 1. Estimator 1 has bias
0.2 and variance 0.01; estimator 2 has bias 0 and variance 0.06. The
cell draws many values of each and measures their mean squared error.

In [ ]:
sim_generator = np.random.default_rng(105)
truth = 1.0
worlds = 200_000
estimator_1 = truth + 0.2 + np.sqrt(0.01) * sim_generator.standard_normal(worlds)
estimator_2 = truth + np.sqrt(0.06) * sim_generator.standard_normal(worlds)
for name, estimates in [("Estimator 1", estimator_1), ("Estimator 2", estimator_2)]:
    bias = estimates.mean() - truth
    variance = estimates.var()
    mean_squared_error = np.mean((estimates - truth) ** 2)
    print(f"{name}: bias^2 + variance = {bias ** 2:.3f} + {variance:.3f}, "
          f"mean squared error = {mean_squared_error:.3f}")

### Weak signal or strong signal: how hard to shrink?

30 predictors, 60 months. In the strong world the predictors explain
half of the outcome’s variance; in the weak world, 5%. For each penalty
$\lambda$ (ridge, which comes later; scikit-learn calls it `alpha`), the
cell averages the squared forecast error over 100 worlds and reports the
best penalty.

In [ ]:
sim_generator = np.random.default_rng(106)
alphas = [0.01, 0.1, 1, 10, 100, 1000, 10000]
for world_name, r2 in [("strong signal (R-squared 50%)", 0.50), ("weak signal (R-squared 5%)", 0.05)]:
    slope = np.sqrt(r2 / (1 - r2) / 30)         # the same true slope on each of the 30 predictors
    average_error = np.zeros(len(alphas))
    for world in range(100):
        X = sim_generator.standard_normal((60, 30))
        y = X.sum(axis=1) * slope + sim_generator.standard_normal(60)
        X_new = sim_generator.standard_normal((500, 30))
        y_new = X_new.sum(axis=1) * slope + sim_generator.standard_normal(500)
        for k, alpha in enumerate(alphas):
            forecast = Ridge(alpha=alpha).fit(X, y).predict(X_new)
            average_error[k] += np.mean((y_new - forecast) ** 2) / 100
    print(f"{world_name}: best λ = {alphas[np.argmin(average_error)]:,}")

# Select: Best Subset, BIC and the 2ᵖ Wall

## Keep a Few: Fit Plus a Fee

- Score each subset of predictors by its fit **plus a fee per
  predictor**:

$$
\begin{aligned}
\text{AIC} &= T \log\!\big(\text{SSR}/T\big) + 2k \\
\text{BIC} &= T \log\!\big(\text{SSR}/T\big) + (\log T)\, k
\end{aligned}
$$

- SSR: sum of squared residuals; $k$: number of predictors in the model
  being scored (out of the $p$ candidates).
- AIC charges 2 per predictor, BIC $\log T$ (6.5 with 672 months), so
  BIC keeps fewer. Read: ISLP 6.1.3.

## The True Model

- Suppose the data came from
  $y = a + b_1 x_1 + \dots + b_p x_p + \varepsilon$.
- **A few** slopes are nonzero; the rest are **exactly** zero.
- The true model: just those few predictors.
- Why want it: it says **which predictors matter**, a question of its
  own in finance.
- And OLS on just those is the best linear unbiased forecast.
- The catch, for forecasting: with our $T$, a model **smaller** than the
  truth can forecast better (segment 3).
- And with returns, no slope may be exactly zero.

## BIC Finds the True Model, Eventually

- If the true model is among the candidates, BIC picks it with
  probability approaching 1 as $T$ grows.
- AIC does not: it tends to keep too many predictors.
- The catches:
  - In samples of our size, BIC often keeps **too few**.
  - Is the true model among the candidates at all?
  - And it must score **every** subset.
- Read: ESL 7.7 (p. 235).

## The Catch: 2ᵖ Regressions

In [ ]:
# Time one OLS fit: 600 months, an intercept and 12 predictors.
generator = np.random.default_rng(0)
X = np.column_stack([np.ones(600), generator.normal(size=(600, 12))])
y = generator.normal(size=600)
start = time.perf_counter()
for repeat in range(2000):
    np.linalg.lstsq(X, y, rcond=None)
seconds_per_fit = (time.perf_counter() - start) / 2000

# Every subset of p predictors: each predictor is in or out, so 2 ** p subsets.
SECONDS_PER_YEAR = 365.25 * 24 * 3600
AGE_OF_UNIVERSE = 13.8e9 * SECONDS_PER_YEAR          # in seconds
p_values = [12, 20, 30, 40, 50, 100]
seconds_needed = [2 ** p * seconds_per_fit for p in p_values]

ticks = [1, SECONDS_PER_YEAR, AGE_OF_UNIVERSE, 1e9 * AGE_OF_UNIVERSE]
tick_names = ["1 second", "1 year", "age of the universe", "a billion × that"]
def in_words(seconds):
    """A duration in the largest sensible unit."""
    years = seconds / SECONDS_PER_YEAR
    if seconds < 10:
        return f"{seconds:.1f} seconds"
    if seconds < 120:
        return f"{seconds:.0f} seconds"
    if seconds < 2 * 86400:
        return f"{seconds / 3600:.0f} hours"
    if years < 1e6:
        return f"{years:,.1f} years" if years < 10 else f"{years:,.0f} years"
    return f"{seconds / AGE_OF_UNIVERSE / 1e6:,.0f} million ×\nthe universe's age"


plt.figure(figsize=(10, 2.7))
plt.bar([str(p) for p in p_values], seconds_needed, color="tab:blue")
for position, seconds in enumerate(seconds_needed):
    plt.text(position, seconds * 3, in_words(seconds), ha="center", va="bottom", fontsize=15)
plt.yscale("log")
plt.yticks(ticks, tick_names)
plt.ylim(0.05, 1e12 * AGE_OF_UNIVERSE)
plt.xlabel("number of candidate predictors p")
plt.ylabel("time to fit every subset")
plt.show()

- One fit takes well under a millisecond, yet all subsets of 40
  predictors take years.
- Refit every month, as week 2’s investor does: × 672.

## Your Turn: Time One Regression

- Time 1,000 fits on your computer, then compute the hours for every
  subset of **30** predictors.

``` python
generator = np.random.default_rng(0)
X = generator.normal(size=(600, 13))
y = generator.normal(size=600)
start = time.perf_counter()
for repeat in range(1000):
    np.linalg.lstsq(X, y, rcond=None)
per_fit = (time.perf_counter() - start) / 1000
hours = 2 ** ____ * per_fit / 3600
print(f"{1e6 * per_fit:.0f} µs a fit; {hours:,.0f} hours")
```

## Stepwise: A Shortcut That Can Miss

- **Greedy:** each step takes the best move now, so it can miss the best
  subset. Forward fits $1 + p(p+1)/2$ models: 466 for 30 predictors.
- Backward needs more observations than predictors.
- Every subset method is all or nothing; shrinkage turns a dial. Read:
  ISLP 6.1.2.

## Check: How Many Regressions?

- Best-subset selection with 30 candidate predictors.
- **Fingers up:** about how many regressions does it fit?
  1.  30
  2.  466
  3.  900
  4.  One billion

# Shrink: Ridge

## Seeing the Minimum, With a Penalty

In [ ]:
# Week 1's picture with one change. Does last month's market return predict this
# month's? x: last month's market excess return; y: this month's. 1970-2025.
both = pd.DataFrame({"x": market.shift(1), "y": market}).loc["1970-01":"2025-12"]
x = (both["x"] - both["x"].mean()) / both["x"].std()    # standardized: mean 0, sd 1
y = both["y"] - both["y"].mean()                         # measured from its average
candidates = np.linspace(-0.002, 0.006, 401)             # candidate slopes b
sse = np.array([np.sum((y - b * x) ** 2) for b in candidates])
penalty = np.sum(x ** 2)                                 # this lambda halves the slope
ols_slope = np.sum(x * y) / np.sum(x ** 2)               # where the blue curve bottoms out
ridge_slope = np.sum(x * y) / (np.sum(x ** 2) + penalty) # where the orange one does

plt.figure(figsize=(10, 2.7))
plt.plot(candidates, sse - sse.min(), color="tab:blue", linewidth=3)
plt.plot(candidates, penalty * candidates ** 2, color="gray", linestyle="--", linewidth=2)
plt.plot(candidates, sse + penalty * candidates ** 2 - sse.min(), color="tab:orange",
         linewidth=3)
plt.axvline(ols_slope, color="tab:blue", linestyle=":")
plt.axvline(ridge_slope, color="tab:orange", linestyle=":")
plt.text(ols_slope, 0.036, " OLS", color="tab:blue")
plt.text(ridge_slope, 0.036, "with penalty ", color="tab:orange", ha="right")
plt.text(0.0047, 0.006, "SSE", color="tab:blue")
plt.text(-0.0019, 0.006, "penalty λb²", color="gray")
plt.text(0.0047, 0.03, "sum", color="tab:orange")
plt.xlim(-0.002, 0.006)
plt.ylim(0, 0.04)
plt.xticks([-0.002, 0, 0.002, 0.004, 0.006], ["−0.002", "0", "0.002", "0.004", "0.006"])
plt.xlabel("candidate slope b")
plt.ylabel("above the minimum SSE")
plt.show()
print(f"OLS slope {ols_slope:.5f}; with the penalty {ridge_slope:.5f}")

- Blue: the SSE, lowest at the OLS slope. Gray: the penalty λb², lowest
  at zero.
- Orange: their sum, lowest in between. The minimum moves toward zero
  and stops short of it.

## Ridge: Least Squares Plus a Penalty

$$
\widehat b^{\,\text{ridge}} = \arg\min_{a,\,b}\;
\sum_{t} \Big(y_t - a - \sum_{j} b_j\, x_{tj}\Big)^2
+ \lambda \sum_{j} b_j^2
$$

- $\lambda = 0$: OLS. A huge $\lambda$: every slope near zero, so the
  forecast is the average of $y$, the **historical mean**.
- No penalty on $a$: the mean is the best constant forecast.
- The books write $\lambda$; scikit-learn calls it `alpha`:
  `Ridge(alpha=...)`. Read: ISLP 6.2.1; ESL 3.4.1.

## One Predictor, One Formula

With $x$ and $y$ measured from their averages:

$$
\widehat b^{\,\text{OLS}} = \frac{\sum_t x_t y_t}{\sum_t x_t^2}
\qquad
\widehat b^{\,\text{ridge}} = \frac{\sum_t x_t y_t}{\sum_t x_t^2 + \lambda}
$$

- So ridge keeps the **fraction**
  $\sum_t x_t^2 / (\sum_t x_t^2 + \lambda)$ of the OLS slope: never
  exactly zero (unless OLS is).
- With more data, $\sum_t x_t^2$ grows, so the same $\lambda$ shrinks
  less.

## Standardize First

In [ ]:
# The same predictor in two units: last month's market return in decimals and in
# percent. OLS does not care; ridge with the same lambda does.
raw_x = market.shift(1).loc["1970-01":"2025-12"]
x_decimals = raw_x.to_frame()               # 0.01 = 1%
x_percent = 100 * raw_x.to_frame()          # 1.0 = 1%
print("1% more last month moves the forecast by (% points):")
for name, predictor, one_percent in [("decimals", x_decimals, 0.01), ("percent", x_percent, 1.0)]:
    ols = LinearRegression().fit(predictor, both["y"]).coef_[0]
    ridge = Ridge(alpha=2.0).fit(predictor, both["y"]).coef_[0]
    print(f"  in {name:8s}: OLS {100 * ols * one_percent:.3f}, ridge {100 * ridge * one_percent:.3f}")

- The penalty charges every slope the same, but a slope’s size depends
  on the units: $b_{\text{percent}} = b_{\text{decimals}} / 100$.
- **Standardize** each predictor, with the fitting months’ mean and
  standard deviation; later months would leak the future:

$$
z_{tj} = \frac{x_{tj} - \bar x_j}{s_j},
\qquad \bar x_j,\ s_j \text{ from the fitting months only}
$$

## When p \> T, OLS Divides by Zero

- 24 months of data, 30 predictors: think of 24 equations with 30
  unknowns: more unknowns than equations.
- OLS looks for the one best set of slopes. Here many sets fit the 24
  months equally well, so there is no one best: the formula has no
  answer, like dividing by zero.
- The ridge penalty breaks the tie: of all the perfect fits, it picks
  the one with the **smallest slopes**.
- So with any $\lambda > 0$, ridge has exactly one answer, even when
  $p > T$.

## Ridge Paths

In [ ]:
# Ridge on the 10-year tracking data: every industry's slope, for each lambda.
# Track the market with the 30 industries, 2016-2025 (120 months), standardized.
X = industries.loc["2016-01":"2025-12"]
X = ((X - X.mean()) / X.std()).to_numpy()           # T = 120 rows, p = 30 columns
y = market.loc["2016-01":"2025-12"].to_numpy()
y = y - y.mean()                                    # from its average: no intercept
penalties = np.logspace(0, 5, 41)                   # lambda from 1 to 100,000
paths = np.array([Ridge(alpha=penalty, fit_intercept=False).fit(X, y).coef_
                  for penalty in penalties])        # one row per lambda
largest = np.argsort(-np.abs(paths[0]))[:3]         # the three largest slopes at lambda = 1
plt.figure(figsize=(10, 3.0))
plt.plot(penalties, paths, color="lightgray", linewidth=1.5)
plt.plot([], [], color="lightgray", linewidth=3, label="the other 27")
for column, color in zip(largest, ["tab:blue", "tab:orange", "tab:green"]):
    plt.plot(penalties, paths[:, column], color=color, linewidth=3,
             label=industry_names[column])
plt.axhline(0, color="black", linewidth=0.8)
plt.xscale("log")
plt.xticks([1, 100, 10000], ["1", "100", "10,000"])
plt.xlim(1, 1e5)
plt.legend(fontsize=16, loc="upper right")
plt.xlabel("λ (log scale)")
plt.ylabel("slope (standardized)")
plt.show()
print("slopes exactly zero at the largest lambda:", int(np.sum(paths[-1] == 0)), "of 30")

- Together the slopes shrink toward zero. A single slope can grow, or
  even switch sign, on the way; none stays at zero.
- Read: ISLP Figure 6.4.

## Check: Double λ

- A ridge regression with **uncorrelated** predictors; $\lambda$
  doubles.
- **Fingers up:** what happens to the slopes?
  1.  The smallest become exactly zero
  2.  Stay the same
  3.  Double
  4.  All move toward zero; none becomes exactly zero

# Shrink to Zero: The Lasso

## The Lasso: Absolute Values Instead of Squares

$$
\widehat b^{\,\text{lasso}} = \arg\min_{a,\,b}\;
\sum_{t} \Big(y_t - a - \sum_{j} b_j\, x_{tj}\Big)^2
+ \lambda \sum_{j} |b_j|
$$

- One change from ridge: the penalty charges $|b_j|$, not $b_j^2$.
- $\lambda = 0$: OLS. A huge $\lambda$: the historical mean, as for
  ridge.
- scikit-learn: `Lasso(alpha=...)`. Tibshirani (1996) named it.

<!-- -->

- Read: ISLP 6.2.2.

## Predict: Which Penalty Gives Exact Zeros?

- Ridge charges $\lambda b^2$; the lasso charges $\lambda |b|$.
- **Predict first:** which can set a slope **exactly** to zero?
  1.  Ridge
  2.  The lasso
  3.  Both
  4.  Neither

## Seeing the Minimum, With a Kink

In [ ]:
# The same SSE as on "Seeing the Minimum, With a Penalty", now with lambda * |b|.
kink = 4.0                                            # this lambda is big enough
lasso_sum = sse + kink * np.abs(candidates)
# The lowest point of the orange curve: OLS moved toward zero by lambda / (2 sum x^2),
# and exactly zero if that would cross zero.
lasso_slope = np.sign(ols_slope) * max(abs(ols_slope) - kink / (2 * np.sum(x ** 2)), 0.0)
plt.figure(figsize=(10, 3.4))
plt.plot(candidates, sse - sse.min(), color="tab:blue", linewidth=3)
plt.plot(candidates, kink * np.abs(candidates), color="gray", linestyle="--", linewidth=2)
plt.plot(candidates, lasso_sum - sse.min(), color="tab:orange", linewidth=3)
plt.axvline(ols_slope, color="tab:blue", linestyle=":")
plt.axvline(lasso_slope, color="tab:orange", linestyle=":")
plt.text(ols_slope, 0.036, " OLS", color="tab:blue")
plt.text(lasso_slope, 0.036, "lasso: exactly 0 ", color="tab:orange", ha="right")
plt.text(0.0047, 0.006, "SSE", color="tab:blue")
plt.text(-0.0019, 0.0115, "penalty λ|b|", color="gray")
plt.text(0.0047, 0.032, "sum", color="tab:orange")
plt.xlim(-0.002, 0.006)
plt.ylim(0, 0.04)
plt.xticks([-0.002, 0, 0.002, 0.004, 0.006], ["−0.002", "0", "0.002", "0.004", "0.006"])
plt.xlabel("candidate slope b")
plt.ylabel("above the minimum SSE")
plt.show()
print(f"OLS slope {ols_slope:.5f}; lasso slope {lasso_slope:.5f}")

- The penalty has a **kink** at zero. When the data’s pull is weaker
  than $\lambda$, the minimum sits exactly on the kink.

## The First Unit Costs

- Move a slope from 0 to a small amount $\varepsilon$:
  - the lasso charges $\lambda\varepsilon$ at once;
  - ridge charges $\lambda\varepsilon^2$: almost nothing.
- So the lasso gives a predictor a slope only if it improves the fit by
  more than $\lambda$ per unit of slope.
- Ridge gives every predictor at least a little.

## A Dead Zone

In [ ]:
# The books' special case: uncorrelated predictors, each scaled so that its sum of
# squares is 1 (X'X = I). Then each slope is shrunk on its own. With the books'
# objectives (SSE + lambda * penalty): ridge keeps 1 / (1 + lambda) of the OLS slope;
# the lasso sets any slope with |OLS| <= lambda / 2 to zero and moves the rest
# lambda / 2 toward zero.
penalty = 1.0
ols_values = np.linspace(-3, 3, 601)
ridge_values = ols_values / (1 + penalty)
lasso_values = np.sign(ols_values) * np.maximum(np.abs(ols_values) - penalty / 2, 0)
plt.figure(figsize=(10, 3.4))
plt.axvspan(-penalty / 2, penalty / 2, color="tab:orange", alpha=0.15)
plt.plot(ols_values, ols_values, color="gray", linestyle="--", linewidth=2)
plt.plot(ols_values, ridge_values, color="tab:blue", linewidth=3)
plt.plot(ols_values, lasso_values, color="tab:orange", linewidth=3)
plt.text(3.05, 3, "OLS", color="gray", va="center")
plt.text(3.05, ridge_values[-1], "ridge", color="tab:blue", va="center")
plt.text(3.05, lasso_values[-1], "lasso", color="tab:orange", va="center")
plt.text(0, -2.2, "lasso: exactly 0\nin this range", color="tab:orange", ha="center")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlim(-3, 3.9)
plt.xticks([-3, -0.5, 0, 0.5, 3], ["−3", "−λ/2", "0", "λ/2", "3"])
plt.xlabel("OLS slope")
plt.ylabel("penalized slope")
plt.show()

- Ridge **scales** every slope down; the lasso **subtracts** a fixed
  amount and stops at zero.
- Read: ISLP Figure 6.10 (the special case).

## The Picture Everyone Draws

- The growing ellipse first touches the diamond at a **corner**, where
  $b_1 = 0$. The circle has no corners. (A bigger λ: a smaller budget.)
- Read: ISLP Figure 6.7; ESL Figure 3.11.

## Corners Catch Whole Wedges

- Every OLS estimate in a shaded wedge lands **on that corner**: one
  slope exactly zero. A whole region, not a lucky point.
- For ridge, only estimates exactly on an axis give a zero.

## In Three Dimensions

- Three slopes: the diamond becomes an octahedron with 6 corners (two
  zeros), 12 edges (one zero), 8 faces (none). Each dot: an OLS
  estimate, moved to its lasso or ridge answer.
- Lasso: many land on a corner (red) or an edge (orange), the rest
  inside a face (blue). Ridge: no zeros.

## More Predictors, More Zeros

- No zero at all means landing **inside** a face: rare with many slopes.
- How many slopes are zero is set by λ, not by $p$.

## Predict: How Many Industries Track the Market?

- The market holds all 30 industries. The lasso picks some; OLS weights
  them (2016–2025).
- **Predict first:** how many for an $R^2$ of 95%?
  1.  One or two
  2.  About four
  3.  About fifteen
  4.  All thirty

## Industries Enter One by One

In [ ]:
# The 10-year tracking data again (standardized X, y from its average). For each lambda,
# the lasso picks industries; OLS on just those gives the tracking R2.
tracking_rows = industries.loc["2016-01":"2025-12"]
X_tracking = (tracking_rows - tracking_rows.mean()) / tracking_rows.std()
y_tracking = market.loc["2016-01":"2025-12"]
r2_by_count = {}                                     # number of industries -> R2
entry_order = []                                     # industries, in order of entry
for alpha in np.logspace(-1.3, -5, 300):             # from a big penalty to a small one
    lasso = Lasso(alpha=alpha, max_iter=100000).fit(X_tracking, y_tracking)
    picked = [name for name, slope in zip(industry_names, lasso.coef_) if slope != 0]
    for name in picked:
        if name not in entry_order:
            entry_order.append(name)
    if picked and len(picked) not in r2_by_count:
        ols = LinearRegression().fit(X_tracking[picked], y_tracking)
        r2_by_count[len(picked)] = 100 * ols.score(X_tracking[picked], y_tracking)
counts = sorted(r2_by_count)
needed_95 = min(count for count in counts if r2_by_count[count] >= 95)
plt.figure(figsize=(10, 3.3))
plt.plot(counts, [r2_by_count[count] for count in counts], color="tab:orange", marker="o",
         linewidth=3)
plt.axhline(95, color="gray", linestyle=":")
plt.text(30.5, 95, "95%", color="gray", va="center")
plt.xlim(0, 33)
plt.ylim(75, 101)
plt.xlabel("industries the lasso keeps")
plt.ylabel("tracking R² (%)")
plt.show()
print("first four in:", ", ".join(entry_order[:4]))

- A handful of industries reach 95%: the lasso finds a small portfolio
  that moves with the market.

## Elastic Net: Some of Each

$$
\text{penalty} = \lambda \Big( m \sum_j |b_j| + (1 - m) \sum_j b_j^2 \Big),
\qquad 0 \le m \le 1
$$

- $m = 1$: the lasso. $m = 0$: ridge. In between: some zeros, and
  correlated predictors stay in or out **together**.
- The lasso alone tends to keep one of two near-twins at random.
- scikit-learn: `ElasticNet(alpha=..., l1_ratio=...)`; `l1_ratio` is
  $m$. Zou & Hastie (2005). Read: ESL 3.4.3.

## Check: The Dead Zone

- Uncorrelated predictors; the lasso’s dead zone is $\pm 0.5$.
- OLS slopes: 0.3, −0.8, 1.2.
- **Fingers up:** what are the lasso’s slopes?
  1.  0, −0.3, 0.7
  2.  0.15, −0.4, 0.6
  3.  0, −0.8, 1.2
  4.  0.3, −0.3, 0.7

# Choosing λ Without Peeking

## Upload, Then Run Setup

Starting a new session here? Upload the three files, run **Setup**, then
this meeting’s cells in order.

# How Hard to Shrink Is a Tuning Parameter

## Every Method Has a Dial

- **Ridge and the lasso:** λ. **Elastic net:** λ and the mix $m$.
- **A shrinkage weight:** Meeting 5’s dial for averages and volatilities
  (read off the answer, with hindsight).
- **Best subset:** how many predictors to keep.
- Later: a tree’s depth, the number of trees (Meetings 7 and 8).
- A **tuning parameter** is set before the fit. The fit itself cannot
  choose it.

## Predict: Choose λ by the Training Error?

- You fit ridge for 50 values of λ and keep the one with the smallest
  error **on the data you fitted**.
- **Predict first:** which λ wins?
  1.  The smallest: λ = 0, OLS
  2.  One in the middle
  3.  The largest
  4.  It depends on the data

## Why the Fit Cannot Set λ

In [ ]:
# A world we control: T = 60, p = 30, true slopes with R^2 = 20%, noise variance 1.
# For each lambda: the error on the 60 fitted rows, and on 2,000 new rows.
generator = np.random.default_rng(21)
T, p = 60, 30
penalties = np.logspace(-1, 4, 26)
true_slopes = generator.normal(size=p)
true_slopes *= np.sqrt(0.25 / np.sum(true_slopes ** 2))        # R^2 = 0.25/1.25 = 20%
training_error = np.zeros(len(penalties))
new_data_error = np.zeros(len(penalties))
for simulation in range(200):
    X = generator.normal(size=(T, p))
    y = X @ true_slopes + generator.normal(size=T)
    X_new = generator.normal(size=(2000, p))
    y_new = X_new @ true_slopes + generator.normal(size=2000)
    for j, penalty in enumerate(penalties):
        ridge = Ridge(alpha=penalty, fit_intercept=False).fit(X, y)   # penalty = lambda
        training_error[j] += np.mean((y - ridge.predict(X)) ** 2) / 200
        new_data_error[j] += np.mean((y_new - ridge.predict(X_new)) ** 2) / 200

plt.figure(figsize=(10, 3.2))
plt.plot(penalties, training_error, color="tab:blue", linewidth=3)
plt.plot(penalties, new_data_error, color="tab:orange", linewidth=3)
plt.plot(penalties[np.argmin(new_data_error)], new_data_error.min(), "o", color="tab:orange",
         markersize=10)
plt.text(0.12, training_error[0] + 0.08, "on the fitted data", color="tab:blue")
plt.text(0.12, new_data_error[0] - 0.22, "on new data", color="tab:orange")
plt.xscale("log")
plt.xticks([0.1, 1, 10, 100, 1000, 10000], ["0.1", "1", "10", "100", "1,000", "10,000"])
plt.xlabel("ridge penalty λ (log scale)")
plt.ylabel("mean squared error")
plt.show()

- On the fitted data the error only rises with λ, so it always picks
  OLS. On new data it is U-shaped: bias against variance.
- We need data the fit has not seen.

## Bloomberg’s Adjusted Beta

In [ ]:
def window_betas(start_year):
    """Each industry's market beta over the 5 years from start_year (60 months)."""
    months = slice(f"{start_year}-01", f"{start_year + 4}-12")
    x = market.loc[months]                           # market excess return
    betas = {}
    for name in industry_names:
        y = industries.loc[months, name]             # industry excess return
        betas[name] = np.cov(x, y)[0, 1] / np.var(x, ddof=1)   # the OLS slope
    return pd.Series(betas)


first_starts = list(range(1970, 2016, 5))            # the ten window pairs, as in Meeting 5
raw_errors, adjusted_errors = [], []
for start in first_starts:
    before, after = window_betas(start), window_betas(start + 5)
    adjusted = 0.67 * before + 0.33 * 1             # Bloomberg's adjustment
    raw_errors.append(np.mean((after - before) ** 2))
    adjusted_errors.append(np.mean((after - adjusted) ** 2))

before, after = window_betas(2010), window_betas(2015)
slope = np.polyfit(before, after, 1)[0]              # fitted line through the 30 points
line = np.linspace(0.3, 1.7, 2)
plt.figure(figsize=(10, 2.3))
plt.scatter(before, after, color="tab:blue", s=50)
plt.plot(line, line, color="black", linestyle="--", linewidth=2)
plt.plot(line, after.mean() + slope * (line - before.mean()), color="tab:orange", linewidth=3)
plt.text(1.72, 1.7, "next = past", va="center")
plt.text(1.72, after.mean() + slope * (1.7 - before.mean()), f"fitted: slope {slope:.2f}",
         color="tab:orange", va="center")
plt.xlim(0.25, 2.35)
plt.xlabel("beta, 2010-14")
plt.ylabel("beta, 2015-19")
plt.show()

- Bloomberg’s terminal reports an **adjusted beta**: 0.67 × the
  estimate + 0.33 × 1. A shrinkage weight someone chose.
- Why: high betas fall, low betas rise (the fitted slope is below 1).
  Over ten window pairs since 1970 the adjusted beta usually misses next
  period’s beta by less.

## Where the 2/3 Came From

- Blume (1971) regressed betas from one period on the betas of the
  period before. The slope: about two thirds, so the weight on 1 is
  about one third.
- That slope is a shrinkage weight, **estimated on past data**, then
  used going forward: a tuning parameter chosen before the fit.
- Next: the same with our 30 industries, using only the window pairs
  that came before.

## Tuning the Beta Weight

- A beta forecast with a dial $w$, the weight on 1:

$$
\widehat\beta^{\,\text{next}} = (1 - w)\,\widehat\beta^{\,\text{past}} + w \cdot 1
$$

- Choose $w$ on the window pairs that came **before** the one being
  forecast: the $w$ with the smallest squared error there.

<!-- -->

- Lowest error at one weight, $w^*$. Score the next pair with it; that
  pair never touches the choice.

## Tuned on the Past

In [ ]:
def shrink(betas, weight):
    """Move each beta toward 1 by `weight` (0: keep it; 1: replace it by 1)."""
    return (1 - weight) * betas + weight * 1.0


pairs = [(window_betas(start), window_betas(start + 5)) for start in range(1930, 2016, 5)]
scored_starts, tuned_weights = [], []
errors = {"raw": [], "Bloomberg": [], "tuned": []}
for k, (before, after) in enumerate(pairs):
    start = 1930 + 5 * k
    if start < 1970:                                 # score the pairs from 1970 on
        continue
    earlier = pairs[:k]                              # only pairs that ended before this one
    change = np.concatenate([(later - past).to_numpy() for past, later in earlier])
    distance = np.concatenate([(1 - past).to_numpy() for past, later in earlier])
    w_tuned = np.sum(change * distance) / np.sum(distance ** 2)   # the regression slope
    scored_starts.append(start)
    tuned_weights.append(w_tuned)
    errors["raw"].append(np.mean((after - before) ** 2))
    errors["Bloomberg"].append(np.mean((after - shrink(before, 0.33)) ** 2))
    errors["tuned"].append(np.mean((after - shrink(before, w_tuned)) ** 2))

plt.figure(figsize=(10, 2.3))
plt.plot(scored_starts, tuned_weights, color="tab:orange", marker="o", linewidth=3)
plt.axhline(0.33, color="gray", linestyle=":")
plt.text(2021, 0.32, "Bloomberg: 0.33", color="gray", va="top")
plt.text(2021, tuned_weights[-1] + 0.02, "tuned on the past", color="tab:orange", va="bottom")
plt.ylim(0, 0.7)
plt.xlim(1968, 2042)
plt.xticks(scored_starts, [f"{start % 100:02d}" for start in scored_starts], fontsize=16)
plt.xlabel("first window of the scored pair: 1970, 1975, ..., 2015")
plt.ylabel("weight on 1")
plt.show()
print("MSE: " + ", ".join(f"{name} {np.mean(v):.4f}" for name, v in errors.items()))

- Chosen from the past, the weight is stable across the pairs;
  Bloomberg’s 0.33 does about as well.
- Tuned on the past it does as well as Bloomberg’s fixed weight; both
  beat the raw betas (MSE: mean squared error).

## Check: Which Is a Tuning Parameter?

- **Fingers up:** which of these is a tuning parameter?
  1.  The OLS slope on the market
  2.  The penalty λ in a ridge regression
  3.  The average return of an industry
  4.  The $R^2$ of a regression

# Train, Validate, Test

## Three Blocks of Time

- Fit on the past, pick λ on the next block, score the winner **once**
  on a block nobody has looked at.
- Why once: the more models compete on a block, the more its winner’s
  score is luck. The test block stays in a vault (ESL 7.2). Read: ISLP
  5.1.1.

## The Validation Curve

In [ ]:
# One sample from the same world: 60 rows to fit, the next 30 to validate.
generator = np.random.default_rng(4)
X = generator.normal(size=(90, p))
y = X @ true_slopes + generator.normal(size=90)
X_fit, y_fit, X_check, y_check = X[:60], y[:60], X[60:], y[60:]
validation_error = []
for penalty in penalties:
    ridge = Ridge(alpha=penalty, fit_intercept=False).fit(X_fit, y_fit)   # penalty = lambda
    validation_error.append(np.mean((y_check - ridge.predict(X_check)) ** 2))
chosen = penalties[np.argmin(validation_error)]

plt.figure(figsize=(10, 3.2))
plt.plot(penalties, new_data_error, color="black", linewidth=3)
plt.plot(penalties, validation_error, color="tab:orange", linewidth=3)
plt.plot(chosen, min(validation_error), "o", color="tab:orange", markersize=10)
plt.text(0.12, new_data_error[0] + 0.05, "truth (new data)", va="bottom")
plt.text(0.12, 0.45, "30-row validation block", color="tab:orange", va="bottom")
plt.xscale("log")
plt.xticks([0.1, 1, 10, 100, 1000, 10000], ["0.1", "1", "10", "100", "1,000", "10,000"])
plt.ylim(0.4, 2.4)
plt.xlabel("ridge penalty λ (log scale)")
plt.ylabel("mean squared error")
plt.show()

- One validation block gives a **noisy** curve. Its pick (orange dot)
  lands near the best λ, not on it.

## Check: Which Block Picks λ?

- Train, validate, test.
- **Fingers up:** which block chooses λ?
  1.  The validation block
  2.  The training block
  3.  The test block
  4.  All three together

# K-Fold Cross-Validation

## Every Block Takes a Turn

- Five rounds; each block validates once. Average the five validation
  errors for every λ, pick the best λ, refit on all the data.
- Read: ISLP 5.1.3.

## The CV Curve

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
# A simulated sparse world: 100 rows, 50 predictors, only 5 of them matter.
generator = np.random.default_rng(12)
X = generator.normal(size=(100, 50))
slopes = np.zeros(50)
slopes[:5] = 0.5
y = X @ slopes + generator.normal(size=100)
model = LassoCV(cv=KFold(5, shuffle=True, random_state=0), max_iter=50000).fit(X, y)
cv_error = model.mse_path_.mean(axis=1)              # one value per lambda
cv_se = model.mse_path_.std(axis=1, ddof=1) / np.sqrt(5)   # its standard error
best = np.argmin(cv_error)
# One standard error: the largest lambda (fewest predictors) within 1 SE of the best.
within = np.nonzero(cv_error <= cv_error[best] + cv_se[best])[0]
one_se = within[np.argmax(model.alphas_[within])]
kept = [int(np.sum(Lasso(alpha=a, max_iter=50000).fit(X, y).coef_ != 0)) for a in model.alphas_]

plt.figure(figsize=(10, 3.0))
plt.fill_between(model.alphas_, cv_error - cv_se, cv_error + cv_se, color="tab:blue", alpha=0.2)
plt.plot(model.alphas_, cv_error, color="tab:blue", linewidth=3)
plt.axvline(model.alphas_[best], color="tab:orange", linestyle="--", linewidth=2)
plt.axvline(model.alphas_[one_se], color="black", linestyle=":", linewidth=2)
plt.text(model.alphas_[best], max(cv_error), "minimum ", color="tab:orange", va="top", ha="right")
plt.text(model.alphas_[one_se], max(cv_error) * 0.93, " one SE", va="top")
plt.xscale("log")
plt.xticks([0.001, 0.01, 0.1], ["0.001", "0.01", "0.1"])
plt.xlabel("lasso penalty λ (log scale); band: ± one standard error")
plt.ylabel("5-fold CV error")
plt.show()
print(f"predictors kept: {kept[best]} at the minimum, {kept[one_se]} at one SE (truth: 5)")

- The **one-standard-error rule**: take the simplest model within one
  standard error of the minimum. Read: ISLP 6.1.3 and 6.2.3.

## How Many Folds?

- $k$ = 5 or 10 is the usual choice: each fit sees 80–90% of the data.
- $k = T$, leave-one-out: $T$ fits, each on almost everything.
- Fewer folds: cheaper, each fit sees less data.
- Read: ISLP 5.1.2–5.1.4.

## Check: Five Folds, 600 Months

- 5-fold cross-validation on 600 months.
- **Fingers up:** how many months does each fit use?
  1.  120
  2.  600
  3.  480
  4.  500

# Time Order

## Shuffled Folds Let the Future Predict the Past

- Blue: fit. Orange: validate. With shuffled folds, years **after** a
  validation year help predict it.
- Read: CASI 12.4.

## Overlapping Returns Leak

- Week 2’s five-year returns: two neighbors share **four** of their five
  years.
- Shuffled folds put a validation year’s neighbors in the fitting data:
  the forecast has seen most of its answer.
- The fix: forward chaining, plus a **gap** of four years between the
  fitting and the validation years.

## Predict: What Did Shuffled CV Deliver?

- Ridge forecasts of five-year returns from 14 annual predictors; λ
  chosen by **shuffled** 5-fold cross-validation.
- Its cross-validation score promised a large positive $R^2$.
- **Predict first:** what did it deliver out of sample?
  1.  About what it promised
  2.  About half of it
  3.  About zero
  4.  Well below zero

## Promised Against Delivered

In [ ]:
def r2(actual, forecast, benchmark):
    """Out-of-sample R2: 1 minus squared error relative to the benchmark's."""
    return 1 - np.sum((actual - forecast) ** 2) / np.sum((actual - benchmark) ** 2)


from sklearn.model_selection import KFold, TimeSeriesSplit
# Annual Goyal-Welch: 14 predictors at the end of year t - 1 (gw_annual has no price
# level, so the ratios use the December S&P 500 level from the monthly file, as in week 2).
december = monthly[monthly.index.month == 12]
price = pd.Series(december["Index"].to_numpy(), index=december.index.year)
x = pd.DataFrame(index=annual.index)
x["dp"] = np.log(annual["d12"]) - np.log(price)          # log dividend-price ratio
x["dy"] = np.log(annual["d12"]) - np.log(price.shift(1))  # log dividend yield
x["ep"] = np.log(annual["e12"]) - np.log(price)          # log earnings-price ratio
x["de"] = np.log(annual["d12"]) - np.log(annual["e12"])  # log payout ratio
for name, column in [("svar", "svar"), ("bm", "b/m"), ("ntis", "ntis"), ("tbl", "tbl"),
                     ("lty", "lty"), ("ltr", "ltr"), ("infl", "infl")]:
    x[name] = annual[column]
x["tms"] = annual["lty"] - annual["tbl"]                 # term spread
x["dfy"] = annual["BAA"] - annual["AAA"]                 # default yield spread
x["dfr"] = annual["corpr"] - annual["ltr"]               # default return spread
table = x.shift(1)                                       # row t: the end of year t - 1
# Target: the premium of years t to t + 4, added up (simple, decimals). Overlapping.
table["target"] = sum(annual["equity"].shift(-j) for j in range(5))
table = table.dropna()
X_all, y_all = table.drop(columns="target").to_numpy(), table["target"].to_numpy()
years = table.index.to_numpy()
alphas = np.logspace(-1, 7, 33)


def ridge_forecasts(X_fit, y_fit, X_new, alpha):
    """Ridge, with predictors standardized on the fitting rows only."""
    mean, sd = X_fit.mean(axis=0), X_fit.std(axis=0)
    Z_fit, Z_new = (X_fit - mean) / sd, (X_new - mean) / sd
    ridge = Ridge(alpha=alpha).fit(Z_fit, y_fit)        # fits the intercept (y's average)
    return ridge.predict(Z_new)


def cross_validate(X, y, splitter):
    """The lambda with the smallest validation error, and the R2 it promises."""
    errors, benchmark = np.zeros(len(alphas)), 0.0
    for fit_rows, check_rows in splitter.split(X):
        for j, alpha in enumerate(alphas):
            forecast = ridge_forecasts(X[fit_rows], y[fit_rows], X[check_rows], alpha)
            errors[j] += np.sum((y[check_rows] - forecast) ** 2)
        benchmark += np.sum((y[check_rows] - y[fit_rows].mean()) ** 2)
    return alphas[np.argmin(errors)], 1 - errors.min() / benchmark


methods = {"shuffled": [KFold(5, shuffle=True, random_state=seed) for seed in range(10)],
           "forward, gap 4": [TimeSeriesSplit(5, gap=4)]}
promised = {name: [] for name in methods}
delivered = {name: [] for name in methods}
rows = np.nonzero(years >= 1965)[0]                      # forecasts for 1965 to 2020
actual = y_all[rows]
benchmark = np.array([y_all[:k - 4].mean() for k in rows])   # targets known by t - 1
for name, splitters in methods.items():
    for splitter in splitters:
        forecasts, promises = [], []
        for k in rows:
            X_fit, y_fit = X_all[:k - 4], y_all[:k - 4]  # rows whose 5 years have ended
            alpha, promise = cross_validate(X_fit, y_fit, splitter)
            forecasts.append(ridge_forecasts(X_fit, y_fit, X_all[k:k + 1], alpha)[0])
            promises.append(promise)
        promised[name].append(100 * np.mean(promises))
        delivered[name].append(100 * r2(actual, np.array(forecasts), benchmark))
summary = {name: (np.median(promised[name]), np.median(delivered[name])) for name in methods}

plt.figure(figsize=(10, 3.0))
for position, (name, (promise, result)) in enumerate(summary.items()):
    plt.bar(position - 0.18, promise, width=0.34, color="gray")
    plt.bar(position + 0.18, result, width=0.34, color="tab:orange")
    plt.text(position - 0.18, promise + 2, f"{promise:+.0f}%", ha="center", va="bottom")
    plt.text(position + 0.18, result + (2 if result > 0 else -3), f"{result:+.0f}%",
             ha="center", va="bottom" if result > 0 else "top")
plt.axhline(0, color="black", linewidth=1)
plt.xticks([0, 1], ["shuffled folds", "forward, 4-year gap"])
plt.text(1.5, 30, "gray: promised\norange: delivered", va="center")
plt.xlim(-0.6, 2.6)
plt.ylim(-60, 65)
plt.ylabel("R² (%)")
plt.show()

- Shuffled folds promised a lot and delivered far below zero. Forward
  chaining with a gap promised far less, and its promise held up much
  better.

## Check: Why Shuffled CV Fails Here

- Shuffled 5-fold cross-validation looks great on overlapping five-year
  returns.
- **Fingers up:** why?
  1.  Five-year returns are normally distributed
  2.  Validation years share returns with fitting years
  3.  Ridge cannot handle overlapping data
  4.  Five folds are too few

# Tune Inside, Test Outside

## Two Loops

- **Inner loop:** on the fitting data only, cross-validate to choose λ.
- **Outer loop:** score the whole recipe, the choice of λ included, on
  data it never touched.
- Report the outer score. This is called nested cross-validation.

## Real Time Is Nested

- Each January: choose λ by forward chaining on the years before.
- Each month: forecast with that λ, using only the past.
- At the end: score all forecasts against the historical mean.
- Week 1’s loop, with a λ-choosing step inside. PS2 builds one.

## Do Industries Lead the Market?

In [ ]:
# Last month's returns of the 30 industries -> this month's market excess return.
# Expanding window from 1926; forecasts 1970-01 to 2025-12; the historical mean as the
# benchmark. Each January, lambda is chosen on the 10 years before: for each of those
# years, fit on all months before it and forecast its 12 months (forward chaining).
data = pd.concat([industries.shift(1), market.rename("market_now")], axis=1).loc["1926-08":]
X_all, y_all = data[industry_names].to_numpy(), data["market_now"].to_numpy()
first = data.index.get_loc(pd.Timestamp("1970-01-01"))
ridge_grid = np.logspace(1, 6, 11)                  # scikit-learn Ridge scale
lasso_grid = np.logspace(-4, -2, 9)                 # scikit-learn Lasso scale


def lasso_forecasts(X_fit, y_fit, X_new, alpha):
    """The lasso, with predictors standardized on the fitting rows only."""
    mean, sd = X_fit.mean(axis=0), X_fit.std(axis=0)
    model = Lasso(alpha=alpha, max_iter=20000).fit((X_fit - mean) / sd, y_fit)
    return model.predict((X_new - mean) / sd)


def choose(X, y, years, this_year, grid, forecasts):
    """The penalty with the smallest error over the 10 years before this_year."""
    errors = np.zeros(len(grid))
    for check_year in range(this_year - 10, this_year):
        fit_rows, check_rows = years < check_year, years == check_year
        for j, penalty in enumerate(grid):
            forecast = forecasts(X[fit_rows], y[fit_rows], X[check_rows], penalty)
            errors[j] += np.sum((y[check_rows] - forecast) ** 2)
    return grid[np.argmin(errors)]


results = {"OLS": [], "ridge": [], "lasso": [], "mean": []}
lasso_kept = []
for t in range(first, len(data)):
    X_fit, y_fit, x_now = X_all[:t], y_all[:t], X_all[t:t + 1]   # months before t only
    if data.index[t].month == 1:                                 # re-choose each January
        years = data.index[:t].year
        ridge_penalty = choose(X_fit, y_fit, years, data.index[t].year, ridge_grid,
                               ridge_forecasts)
        lasso_penalty = choose(X_fit, y_fit, years, data.index[t].year, lasso_grid,
                               lasso_forecasts)
    results["OLS"].append(LinearRegression().fit(X_fit, y_fit).predict(x_now)[0])
    results["ridge"].append(ridge_forecasts(X_fit, y_fit, x_now, ridge_penalty)[0])
    mean, sd = X_fit.mean(axis=0), X_fit.std(axis=0)
    lasso = Lasso(alpha=lasso_penalty, max_iter=20000).fit((X_fit - mean) / sd, y_fit)
    results["lasso"].append(lasso.predict((x_now - mean) / sd)[0])
    lasso_kept.append(int(np.sum(lasso.coef_ != 0)))
    results["mean"].append(y_fit.mean())

actual = y_all[first:]
mean_forecast = np.array(results["mean"])
scores = {name: 100 * r2(actual, np.array(results[name]), mean_forecast)
          for name in ["OLS", "ridge", "lasso"]}
plt.figure(figsize=(10, 2.5))
for name, color, nudge in [("OLS", "gray", 0), ("ridge", "tab:blue", 0.8),
                           ("lasso", "tab:orange", -0.8)]:
    gain = np.cumsum((actual - mean_forecast) ** 2 - (actual - np.array(results[name])) ** 2)
    plt.plot(data.index[first:], 100 * gain, color=color, linewidth=3)
    plt.text(data.index[-1], 100 * gain[-1] + nudge, f" {name}", color=color, va="center")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlim(data.index[first], data.index[-1] + pd.DateOffset(years=8))
plt.xticks([pd.Timestamp(f"{year}-01-01") for year in range(1970, 2030, 10)],
           [str(year) for year in range(1970, 2030, 10)])
plt.ylabel("running gain over the\nhistorical mean (× 100)")
plt.show()
print("R2 out of sample: " + ", ".join(f"{name} {value:+.2f}%"
                                       for name, value in scores.items()))
print(f"lasso keeps no industry in {np.mean(np.array(lasso_kept) == 0):.0%} of months")

- Shrinkage cuts most of OLS’s loss, but nothing here beats the
  historical mean.

## Check: Which Data Choose λ for January 2000?

- A real-time evaluation forecasts January 2000.
- **Fingers up:** which data may choose its λ?
  1.  All the data, 1926–2025
  2.  Data through December 1999 only
  3.  Data from 2000 on
  4.  The months being scored

# Close

## Week 3 in One Slide

- Many predictors, few observations: OLS fits noise, and with $p \geq T$
  it cannot even choose.
- Shrink: a little bias for much less variance. Ridge makes slopes
  smaller; the lasso sets some to zero.
- How hard to shrink is a tuning parameter: choose it on data the fit
  has not seen.
- With time series: validate on the future of the fitting data, leave
  gaps when returns overlap, and choose λ only on data the fit has not
  seen.

## Not in This Deck: Summarize

- Week 2’s fourth way out: summarize many predictors in a few factors
  (principal components, partial least squares). Meetings 9 and 10.
- Next: curves and interactions. Trees, Meetings 7 and 8.

## Readings

**Meeting 5: ridge and lasso.**

- Required: Hastie, Tibshirani & Friedman, *The Elements of Statistical
  Learning*, ch. 3, sections 3.3–3.4.
- Gentler: James, Witten, Hastie, Tibshirani & Taylor (2023), *ISLP*,
  6.1–6.2 and 6.4.
- Optional: Efron & Hastie (2016), *Computer Age Statistical Inference*,
  7.1–7.3.

**Meeting 6: cross-validation and overfitting.**

- Required: *The Elements of Statistical Learning*, ch. 7, sections
  7.1–7.3 and 7.10.
- Gentler: *ISLP* 2.2 and 5.1.
- Optional: *Computer Age Statistical Inference* 12.4.